# CalibrateQwen: unified evaluation and technical report

We use this notebook to compare the unmodified Qwen3.5-4B base model with the completed off-policy and on-policy CalibrateQwen checkpoints. The notebook recovers checkpoints from Tinker, samples matched examples, verifies every prediction artifact, fits calibration parameters on validation data, evaluates held-out data, and generates a compact report.

The protocol keeps three quantities distinct:

1. **Answer accuracy** measures whether the selected option matches the gold label.
2. **Verbal confidence** is the confidence value emitted inside the model's structured JSON response.
3. **Answer-token probability** is obtained by scoring every option label under a dedicated answer-only prompt and normalizing those scores.

We fit temperature scaling and the abstention threshold independently for each model using validation predictions. We freeze those parameters before evaluating test, out-of-domain, and option-shuffled examples. This separation prevents test-label leakage.


## 1. Start from a clean Colab runtime

This cell clones the repository when necessary and installs its pinned dependencies. The requirements constrain NumPy to the ABI range used by current Colab images and pin pandas to Colab's supported release. When Colab reports that imported packages changed, restart the runtime once and continue from the next cell.


In [ ]:
from pathlib import Path

REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha

%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q -r requirements.txt


## 2. Load credentials and choose the report scope

We read credentials from Colab Secrets. The Hugging Face secret in this project is named `HF_WRITE_ACCESS`; the repository code reads it through the standard `HF_TOKEN` environment variable. The notebook never prints either credential.

`RUN_MODE` controls cost and runtime:

- `smoke` evaluates 25 records per split and validates the complete pipeline.
- `core` evaluates 250 records per split and is suitable for an initial three-model report.
- `full` evaluates every available record and should be used for final reported numbers.

`RUN_OOD` and `RUN_ROBUSTNESS` control the extended evaluations. Option-shuffled evaluation uses two perturbations for each selected in-domain example.


In [ ]:
import os
from google.colab import userdata

def colab_secret(name):
    value = userdata.get(name)
    if value is None or not value.strip():
        raise RuntimeError(f'Add {name} to the Colab Secrets panel and enable notebook access.')
    return value.strip()

os.environ['TINKER_API_KEY'] = colab_secret('TINKER_API_KEY')
os.environ['HF_TOKEN'] = colab_secret('HF_WRITE_ACCESS')

RUN_MODE = 'core'  # Choose: 'smoke', 'core', or 'full'
RUN_OOD = True
RUN_ROBUSTNESS = True
PERSIST_TO_DRIVE = True

LIMIT_BY_MODE = {'smoke': 25, 'core': 250, 'full': None}
if RUN_MODE not in LIMIT_BY_MODE:
    raise ValueError(f'Unknown RUN_MODE: {RUN_MODE}')
EVALUATION_LIMIT = LIMIT_BY_MODE[RUN_MODE]

DATASET_REPO = 'ritwikraha/calibrate-qwen-curated'
BASE_MODEL = 'Qwen/Qwen3.5-4B'
RENDERER = 'qwen3_5_disable_thinking'
TARGET_COVERAGE = 0.80
PERMUTATIONS_PER_EXAMPLE = 2

print({
    'run_mode': RUN_MODE,
    'records_per_core_split': EVALUATION_LIMIT,
    'run_ood': RUN_OOD,
    'run_robustness': RUN_ROBUSTNESS,
    'persist_to_drive': PERSIST_TO_DRIVE,
})


### Persisting the report

Colab deletes `/content` when a runtime ends. We therefore store predictions, calibration objects, metrics, plots, tables, and the final manifest in Google Drive by default. Sampling is resumable by record ID, so rerunning the notebook continues incomplete files instead of paying for completed requests again.


In [ ]:
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPORT_ROOT = Path('/content/drive/MyDrive/calibrate_qwen_results/unified_report')
else:
    REPORT_ROOT = Path('/content/calibrate_qwen_results/unified_report')

PREDICTION_DIR = REPORT_ROOT / 'predictions'
CALIBRATION_DIR = REPORT_ROOT / 'calibration'
METRICS_DIR = REPORT_ROOT / 'metrics'
FIGURE_DIR = REPORT_ROOT / 'figures'
TABLE_DIR = REPORT_ROOT / 'tables'

for directory in [PREDICTION_DIR, CALIBRATION_DIR, METRICS_DIR, FIGURE_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f'Report root: {REPORT_ROOT}')


## 3. Recover and verify checkpoints from Tinker

The local `checkpoints.jsonl` files belong to the Colab runtime that performed training. Tinker stores the actual checkpoints remotely. We query the authenticated account, display all completed sampler checkpoints, and resolve the two known training-run IDs explicitly.

Evaluation requires a **sampler checkpoint** such as `sampler_weights/final`. A training-state checkpoint under `weights/` is intended for resuming optimization. We verify each selected sampler path through Tinker before submitting evaluation requests.


In [ ]:
from datetime import datetime, timezone
import json
import pandas as pd
import tinker

OFF_POLICY_RUN_ID = '7efd2713-b159-5099-aa88-659b038e3cc3:train:0'
ON_POLICY_RUN_ID = 'e45e676a-8586-5d5d-a00f-2c4527c33929:train:0'

service_client = tinker.ServiceClient()
rest_client = service_client.create_rest_client()

runs_response = rest_client.list_training_runs(limit=100).result()
checkpoint_rows = []
for run in runs_response.training_runs:
    checkpoint = run.last_sampler_checkpoint
    if checkpoint is None:
        continue
    checkpoint_rows.append({
        'run_id': run.training_run_id,
        'base_model': run.base_model,
        'last_activity': run.last_request_time,
        'sampler_time': checkpoint.time,
        'checkpoint_id': checkpoint.checkpoint_id,
        'expires_at': checkpoint.expires_at,
        'metadata': json.dumps(run.user_metadata or {}),
        'sampler_path': checkpoint.tinker_path,
    })

checkpoint_table = (
    pd.DataFrame(checkpoint_rows)
    .sort_values('sampler_time', ascending=False)
    .reset_index(drop=True)
)
display(checkpoint_table)


In [ ]:
def final_sampler_checkpoint(run_id):
    response = rest_client.list_checkpoints(run_id).result()
    candidates = []
    for checkpoint in response.checkpoints:
        checkpoint_type = getattr(
            checkpoint.checkpoint_type,
            'value',
            str(checkpoint.checkpoint_type),
        ).lower()
        if 'sampler' in checkpoint_type or '/sampler_weights/' in checkpoint.tinker_path:
            candidates.append(checkpoint)

    if not candidates:
        raise RuntimeError(f'No sampler checkpoint is available for {run_id}.')

    selected = max(
        candidates,
        key=lambda item: (
            item.checkpoint_id == 'sampler_weights/final'
            or item.tinker_path.endswith('/sampler_weights/final'),
            item.time,
        ),
    )
    if selected.expires_at is not None:
        expiry = selected.expires_at
        if expiry.tzinfo is None:
            expiry = expiry.replace(tzinfo=timezone.utc)
        if expiry <= datetime.now(timezone.utc):
            raise RuntimeError(f'Checkpoint expired at {expiry}: {selected.tinker_path}')
    return selected.tinker_path

OFF_POLICY_CHECKPOINT = final_sampler_checkpoint(OFF_POLICY_RUN_ID)
ON_POLICY_CHECKPOINT = final_sampler_checkpoint(ON_POLICY_RUN_ID)

MODELS = {
    'base': None,
    'off_policy': OFF_POLICY_CHECKPOINT,
    'on_policy': ON_POLICY_CHECKPOINT,
}

assert OFF_POLICY_CHECKPOINT != ON_POLICY_CHECKPOINT
assert '/sampler_weights/' in OFF_POLICY_CHECKPOINT
assert '/sampler_weights/' in ON_POLICY_CHECKPOINT

# This call confirms that the account can resolve each remote path.
for name, checkpoint_path in MODELS.items():
    if checkpoint_path is not None:
        info = rest_client.get_weights_info_by_tinker_path(checkpoint_path).result()
        print(f'{name}: verified {checkpoint_path} ({info.base_model})')
    else:
        print(f'{name}: verified base model {BASE_MODEL}')


## 4. Verify dataset access and construct a matched sampling plan

We inspect every requested Hugging Face configuration before spending Tinker credits. The validation and in-domain test sets come from `calibrated_mcq`. The out-of-domain split is `test_ood`. The `option_shuffled` configuration contains two deterministic permutations per in-domain test example.

A valid comparison requires the same record IDs for all three models. The verification stage after sampling enforces this requirement.


In [ ]:
from datasets import load_dataset

hf_token = os.environ['HF_TOKEN']
dataset_specs = [
    ('validation', 'calibrated_mcq', 'validation'),
    ('test', 'calibrated_mcq', 'test'),
]
if RUN_OOD:
    dataset_specs.append(('ood', 'calibrated_mcq', 'test_ood'))
if RUN_ROBUSTNESS:
    dataset_specs.append(('shuffled', 'option_shuffled', 'test'))

dataset_sizes = {}
for tag, config_name, split_name in dataset_specs:
    split = load_dataset(
        DATASET_REPO,
        config_name,
        split=split_name,
        token=hf_token,
    )
    dataset_sizes[tag] = len(split)
    print(f'{tag:10s} {config_name:18s} {split_name:12s} {len(split):5d} records')

assert dataset_sizes['validation'] > 0
assert dataset_sizes['test'] > 0


In [ ]:
def expected_count(tag, limit):
    return dataset_sizes[tag] if limit is None else min(limit, dataset_sizes[tag])

core_limit = EVALUATION_LIMIT
shuffled_limit = (
    None if EVALUATION_LIMIT is None
    else PERMUTATIONS_PER_EXAMPLE * EVALUATION_LIMIT
)

split_plan = [
    {'tag': 'validation', 'dataset_config': 'calibrated_mcq', 'split': 'validation', 'limit': core_limit},
    {'tag': 'test', 'dataset_config': 'calibrated_mcq', 'split': 'test', 'limit': core_limit},
]
if RUN_OOD:
    split_plan.append(
        {'tag': 'ood', 'dataset_config': 'calibrated_mcq', 'split': 'test_ood', 'limit': core_limit}
    )
if RUN_ROBUSTNESS:
    split_plan.append(
        {'tag': 'shuffled', 'dataset_config': 'option_shuffled', 'split': 'test', 'limit': shuffled_limit}
    )

jobs = []
for model_name, checkpoint_path in MODELS.items():
    for split_spec in split_plan:
        path = PREDICTION_DIR / f'{model_name}_{split_spec["tag"]}_raw.jsonl'
        jobs.append({
            'model': model_name,
            'checkpoint_path': checkpoint_path,
            'output_path': path,
            **split_spec,
        })

sampling_plan = pd.DataFrame([
    {
        'model': job['model'],
        'dataset_config': job['dataset_config'],
        'split': job['split'],
        'expected_records': expected_count(job['tag'], job['limit']),
        'resume_file': str(job['output_path']),
    }
    for job in jobs
])
display(sampling_plan)
print(f'Total requested model-example evaluations: {sampling_plan.expected_records.sum():,}')


## 5. Sample every model

This is the principal paid stage. Each example produces a structured response and separate answer-label scores. The sampler uses temperature `0.2`, a 192-token response limit, and eight concurrent requests. Raw files contain the source record, checkpoint identity, parsed response, original completion, normalized option probabilities, and selected-answer probability.

The sampling function resumes by stable record ID. Keep the same report directory when reconnecting to an interrupted Colab runtime.


In [ ]:
from evaluation.sample_model import SamplingConfig, sample_dataset

for job_number, job in enumerate(jobs, start=1):
    print(
        f'[{job_number}/{len(jobs)}] '
        f'{job["model"]} on {job["tag"]} -> {job["output_path"]}'
    )
    await sample_dataset(SamplingConfig(
        output_path=str(job['output_path']),
        repo_id=DATASET_REPO,
        dataset_config=job['dataset_config'],
        split=job['split'],
        model_name=BASE_MODEL,
        checkpoint_path=job['checkpoint_path'],
        renderer_name=RENDERER,
        confidence_format='numeric',
        temperature=0.2,
        max_tokens=192,
        concurrency=8,
        limit=job['limit'],
        score_options=True,
        resume=True,
    ))


### Verify raw prediction artifacts

We verify six invariants before computing metrics:

1. Every planned file exists and contains the expected number of records.
2. IDs are unique inside each file.
3. All records contain normalized option probabilities.
4. Each trained-model artifact names the checkpoint selected above.
5. Validation, test, and OOD ID sets match across models.
6. The shuffled subset joins back to the evaluated in-domain parent IDs.

A failed assertion indicates an incomplete or mixed artifact. Resolve that failure before interpreting plots.


In [ ]:
from data.common import read_jsonl, write_jsonl
import math

raw_records = {}
verification_rows = []

for job in jobs:
    key = (job['model'], job['tag'])
    records = list(read_jsonl(job['output_path']))
    expected = expected_count(job['tag'], job['limit'])
    ids = [str(record['id']) for record in records]

    assert len(records) == expected, f'{key}: expected {expected}, found {len(records)}'
    assert len(ids) == len(set(ids)), f'{key}: duplicate IDs found'

    probability_errors = 0
    checkpoint_errors = 0
    for record in records:
        probabilities = record.get('option_probabilities')
        if not isinstance(probabilities, dict) or not math.isclose(
            sum(float(value) for value in probabilities.values()),
            1.0,
            rel_tol=1e-6,
            abs_tol=1e-6,
        ):
            probability_errors += 1
        actual_checkpoint = (record.get('model') or {}).get('checkpoint_path')
        if actual_checkpoint != job['checkpoint_path']:
            checkpoint_errors += 1

    assert probability_errors == 0, f'{key}: {probability_errors} probability errors'
    assert checkpoint_errors == 0, f'{key}: {checkpoint_errors} checkpoint mismatches'
    raw_records[key] = records
    verification_rows.append({
        'model': job['model'],
        'split': job['tag'],
        'records': len(records),
        'unique_ids': len(set(ids)),
        'probability_errors': probability_errors,
        'checkpoint_errors': checkpoint_errors,
        'status': 'verified',
    })

for tag in ['validation', 'test'] + (['ood'] if RUN_OOD else []):
    reference_ids = {str(record['id']) for record in raw_records[('base', tag)]}
    for model_name in MODELS:
        candidate_ids = {str(record['id']) for record in raw_records[(model_name, tag)]}
        assert candidate_ids == reference_ids, f'{tag}: ID mismatch for {model_name}'

if RUN_ROBUSTNESS:
    for model_name in MODELS:
        base_ids = {str(record['id']) for record in raw_records[(model_name, 'test')]}
        parent_ids = {
            str(record.get('parent_id'))
            for record in raw_records[(model_name, 'shuffled')]
        }
        matched_parents = base_ids & parent_ids
        assert matched_parents == base_ids, (
            f'{model_name}: only {len(matched_parents)}/{len(base_ids)} test parents '
            'have option-order perturbations'
        )
        parent_counts = {parent_id: 0 for parent_id in base_ids}
        for record in raw_records[(model_name, 'shuffled')]:
            parent_id = str(record.get('parent_id'))
            if parent_id in parent_counts:
                parent_counts[parent_id] += 1
        assert set(parent_counts.values()) == {PERMUTATIONS_PER_EXAMPLE}, (
            f'{model_name}: expected {PERMUTATIONS_PER_EXAMPLE} perturbations per parent'
        )
        print(f'{model_name}: {len(matched_parents)}/{len(base_ids)} test parents have perturbations')

verification_table = pd.DataFrame(verification_rows)
display(verification_table)


## 6. Fit validation calibration and freeze it

For each model, we fit one scalar temperature by minimizing multiclass negative log-likelihood on validation option probabilities. We then select the confidence threshold that answers approximately 80 percent of validation examples. Both values are serialized before test metrics are computed.

Temperature values near `1.0` indicate that the option distribution already has an appropriate global sharpness. Values above `1.0` soften overconfident distributions. Values below `1.0` sharpen underconfident distributions. The abstention threshold is model-specific because score distributions can differ even when accuracy is similar.


In [ ]:
from evaluation.calibrate import apply_calibration, fit_calibration

calibrations = {}
calibration_rows = []

for model_name in MODELS:
    calibration = fit_calibration(
        raw_records[(model_name, 'validation')],
        target_coverage=TARGET_COVERAGE,
    )
    calibrations[model_name] = calibration
    calibration_path = CALIBRATION_DIR / f'{model_name}.json'
    calibration_path.write_text(json.dumps(calibration, indent=2) + '\n', encoding='utf-8')

    abstention = calibration['abstention']
    calibration_rows.append({
        'model': model_name,
        'temperature': calibration['temperature'],
        'validation_nll': calibration['validation_multiclass_nll'],
        'threshold': abstention['threshold'],
        'validation_coverage': abstention['validation_coverage'],
        'validation_selective_accuracy': abstention['validation_selective_accuracy'],
        'validation_risk': abstention['validation_risk'],
    })

calibration_table = pd.DataFrame(calibration_rows).set_index('model')
display(calibration_table.style.format('{:.4f}'))


## 7. Compute held-out metrics

We retain three metric views for the in-domain test set:

- **Verbal** uses the confidence emitted in the model's JSON response.
- **Token raw** uses the normalized probability of the model's selected answer before temperature scaling.
- **Token calibrated** applies the validation temperature and validation abstention threshold.

Accuracy remains identical across these views because calibration changes confidence rather than answer labels. ECE, Brier score, NLL, AURC, and selective accuracy can change substantially.


In [ ]:
from evaluation.metrics import evaluate_records

metric_sets = {}
calibrated_records = {}

evaluation_tags = ['test'] + (['ood'] if RUN_OOD else [])
for model_name in MODELS:
    for tag in evaluation_tags:
        records = raw_records[(model_name, tag)]
        calibrated = [
            apply_calibration(record, calibrations[model_name])
            for record in records
        ]
        calibrated_records[(model_name, tag)] = calibrated
        calibrated_path = PREDICTION_DIR / f'{model_name}_{tag}_calibrated.jsonl'
        write_jsonl(calibrated, calibrated_path)

        views = {
            'verbal': evaluate_records(records, confidence_source='verbal'),
            'token_raw': evaluate_records(records, confidence_source='answer_probability'),
            'token_calibrated': evaluate_records(calibrated, confidence_source='answer_probability'),
        }
        for view_name, metrics in views.items():
            metric_sets[(model_name, tag, view_name)] = metrics
            metric_path = METRICS_DIR / f'{model_name}_{tag}_{view_name}.json'
            metric_path.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')

summary_rows = []
for model_name in MODELS:
    verbal = metric_sets[(model_name, 'test', 'verbal')]
    token_raw = metric_sets[(model_name, 'test', 'token_raw')]
    calibrated = metric_sets[(model_name, 'test', 'token_calibrated')]
    selective = calibrated['model_abstention']
    summary_rows.append({
        'model': model_name,
        'records': calibrated['records'],
        'accuracy': calibrated['accuracy'],
        'format_validity': calibrated['format_validity'],
        'verbal_ece': verbal['ece'],
        'token_ece_raw': token_raw['ece'],
        'token_ece_calibrated': calibrated['ece'],
        'multiclass_brier': calibrated['multiclass_brier'],
        'multiclass_nll': calibrated['multiclass_nll'],
        'aurc': calibrated['aurc'],
        'mean_error_confidence': calibrated['mean_error_confidence'],
        'test_coverage_at_val_threshold': selective['coverage'],
        'selective_accuracy': selective['selective_accuracy'],
    })

summary_table = pd.DataFrame(summary_rows).set_index('model')
summary_table.to_csv(TABLE_DIR / 'in_domain_summary.csv')
display(summary_table.style.format({
    column: '{:.4f}'
    for column in summary_table.columns
    if column != 'records'
}))


### Verification checklist

The following checks make the comparison mechanically auditable. We expect identical record counts, 80 percent validation coverage up to rounding, finite metrics, and unchanged accuracy after calibration. Test coverage is allowed to differ from 80 percent because the threshold was selected on validation data.


In [ ]:
audit_rows = []
expected_test_records = expected_count('test', core_limit)

for model_name in MODELS:
    verbal = metric_sets[(model_name, 'test', 'verbal')]
    raw = metric_sets[(model_name, 'test', 'token_raw')]
    calibrated = metric_sets[(model_name, 'test', 'token_calibrated')]
    validation_coverage = calibrations[model_name]['abstention']['validation_coverage']

    checks = {
        'expected_test_count': calibrated['records'] == expected_test_records,
        'accuracy_invariant': math.isclose(raw['accuracy'], calibrated['accuracy']),
        'finite_ece': math.isfinite(calibrated['ece']),
        'finite_nll': math.isfinite(calibrated['multiclass_nll']),
        'validation_coverage_near_target': abs(validation_coverage - TARGET_COVERAGE) <= 1 / max(1, len(raw_records[(model_name, 'validation')])),
        'format_rate_in_range': 0.0 <= verbal['format_validity'] <= 1.0,
    }
    for check_name, passed in checks.items():
        audit_rows.append({'model': model_name, 'check': check_name, 'passed': bool(passed)})

audit_table = pd.DataFrame(audit_rows)
display(audit_table.pivot(index='check', columns='model', values='passed'))
assert audit_table['passed'].all(), 'At least one report verification check failed.'
print('All report verification checks passed.')


## 8. Visual comparison

We place metrics with the same interpretation on shared axes. Higher values are preferred for accuracy, format validity, and selective accuracy. Lower values are preferred for ECE, Brier score, NLL, AURC, and confidence on errors.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

COLORS = {
    'base': '#386CB0',
    'off_policy': '#D95F5F',
    'on_policy': '#2A9D8F',
}
DISPLAY_NAMES = {
    'base': 'Base',
    'off_policy': 'Off-policy',
    'on_policy': 'On-policy',
}

plot_specs = [
    ('accuracy', 'Accuracy', True),
    ('format_validity', 'Structured-output validity', True),
    ('token_ece_calibrated', 'Calibrated ECE', False),
    ('multiclass_nll', 'Calibrated multiclass NLL', False),
    ('aurc', 'Area under risk-coverage', False),
    ('selective_accuracy', 'Selective accuracy', True),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
model_order = list(MODELS)
for ax, (column, title, higher_is_better) in zip(axes.flat, plot_specs):
    values = [summary_table.loc[name, column] for name in model_order]
    bars = ax.bar(
        [DISPLAY_NAMES[name] for name in model_order],
        values,
        color=[COLORS[name] for name in model_order],
        width=0.68,
    )
    ax.set_title(title)
    ax.set_ylabel('Higher is better' if higher_is_better else 'Lower is better')
    ax.grid(axis='y', alpha=0.22)
    upper = max(values) * 1.22 if max(values) > 0 else 1.0
    ax.set_ylim(0, max(upper, 0.05))
    ax.bar_label(bars, labels=[f'{value:.3f}' for value in values], padding=3, fontsize=9)

fig.suptitle('CalibrateQwen held-out comparison', fontsize=16)
fig.tight_layout()
comparison_path = FIGURE_DIR / 'model_comparison.png'
fig.savefig(comparison_path, dpi=180, bbox_inches='tight')
plt.show()
print(comparison_path)


### Reliability diagrams

A reliability curve compares mean predicted confidence with empirical accuracy inside confidence bins. Points on the diagonal are calibrated. Points below the diagonal indicate overconfidence. Empty bins are omitted because they contain no observations.


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 6.2))
ax.plot([0, 1], [0, 1], linestyle='--', color='#666666', linewidth=1.2, label='Ideal')

for model_name in MODELS:
    bins = [
        item for item in metric_sets[(model_name, 'test', 'token_calibrated')]['calibration_bins']
        if item['count'] > 0
    ]
    ax.plot(
        [item['confidence'] for item in bins],
        [item['accuracy'] for item in bins],
        marker='o',
        linewidth=2.2,
        color=COLORS[model_name],
        label=DISPLAY_NAMES[model_name],
    )

ax.set(
    xlabel='Mean calibrated answer probability',
    ylabel='Empirical accuracy',
    xlim=(0, 1),
    ylim=(0, 1),
    title='Reliability on the in-domain test set',
)
ax.grid(alpha=0.22)
ax.legend()
fig.tight_layout()
reliability_path = FIGURE_DIR / 'reliability_comparison.png'
fig.savefig(reliability_path, dpi=180, bbox_inches='tight')
plt.show()
print(reliability_path)


### Risk-coverage curves

We sort predictions by calibrated confidence and answer progressively more examples. Coverage is the answered fraction. Risk is the error rate among answered examples. A stronger uncertainty estimate keeps risk low as coverage increases, so curves closer to the lower-right boundary are preferred. AURC summarizes the complete curve.


In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 6.2))
for model_name in MODELS:
    curve = metric_sets[(model_name, 'test', 'token_calibrated')]['risk_coverage']
    ax.plot(
        [point['coverage'] for point in curve],
        [point['risk'] for point in curve],
        linewidth=2.2,
        color=COLORS[model_name],
        label=f'{DISPLAY_NAMES[model_name]} (AURC={summary_table.loc[model_name, "aurc"]:.3f})',
    )

ax.set(
    xlabel='Coverage',
    ylabel='Selective risk',
    xlim=(0, 1),
    ylim=(0, None),
    title='Risk-coverage on the in-domain test set',
)
ax.grid(alpha=0.22)
ax.legend()
fig.tight_layout()
risk_path = FIGURE_DIR / 'risk_coverage_comparison.png'
fig.savefig(risk_path, dpi=180, bbox_inches='tight')
plt.show()
print(risk_path)


### Source-level analysis

Aggregate metrics can hide domain-specific regressions. We therefore report calibrated accuracy, ECE, and AURC for each source represented in the in-domain test set. Small source slices should be interpreted with their record count.


In [ ]:
source_rows = []
for model_name in MODELS:
    by_source = metric_sets[(model_name, 'test', 'token_calibrated')]['by_source']
    for source, metrics in by_source.items():
        source_rows.append({
            'model': model_name,
            'source': source,
            'records': metrics['records'],
            'accuracy': metrics['accuracy'],
            'ece': metrics['ece'],
            'aurc': metrics['aurc'],
            'format_validity': metrics['format_validity'],
        })

source_table = pd.DataFrame(source_rows)
source_table.to_csv(TABLE_DIR / 'source_breakdown.csv', index=False)
display(source_table.sort_values(['source', 'model']).style.format({
    'accuracy': '{:.4f}',
    'ece': '{:.4f}',
    'aurc': '{:.4f}',
    'format_validity': '{:.4f}',
}))


## 9. Out-of-domain evaluation

We apply each model's frozen validation calibration to CommonsenseQA without refitting. The in-domain to OOD change measures domain transfer. We compare both answer quality and uncertainty quality because a stable accuracy can coexist with degraded calibration.


In [ ]:
if RUN_OOD:
    domain_rows = []
    for model_name in MODELS:
        for tag, display_tag in [('test', 'In-domain'), ('ood', 'OOD')]:
            metrics = metric_sets[(model_name, tag, 'token_calibrated')]
            domain_rows.append({
                'model': model_name,
                'domain': display_tag,
                'records': metrics['records'],
                'accuracy': metrics['accuracy'],
                'ece': metrics['ece'],
                'multiclass_nll': metrics['multiclass_nll'],
                'aurc': metrics['aurc'],
                'format_validity': metrics['format_validity'],
            })

    domain_table = pd.DataFrame(domain_rows)
    domain_table.to_csv(TABLE_DIR / 'domain_transfer.csv', index=False)
    display(domain_table.style.format({
        'accuracy': '{:.4f}',
        'ece': '{:.4f}',
        'multiclass_nll': '{:.4f}',
        'aurc': '{:.4f}',
        'format_validity': '{:.4f}',
    }))

    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6))
    for ax, metric, title in zip(
        axes,
        ['accuracy', 'ece', 'aurc'],
        ['Accuracy', 'Calibration error', 'Risk-coverage area'],
    ):
        x = np.arange(len(MODELS))
        width = 0.34
        for offset, domain in [(-width / 2, 'In-domain'), (width / 2, 'OOD')]:
            values = [
                domain_table[
                    (domain_table.model == model_name) & (domain_table.domain == domain)
                ][metric].iloc[0]
                for model_name in MODELS
            ]
            ax.bar(x + offset, values, width, label=domain)
        ax.set_xticks(x, [DISPLAY_NAMES[name] for name in MODELS])
        ax.set_title(title)
        ax.grid(axis='y', alpha=0.22)
    axes[0].legend()
    fig.suptitle('In-domain and out-of-domain transfer')
    fig.tight_layout()
    domain_path = FIGURE_DIR / 'domain_transfer.png'
    fig.savefig(domain_path, dpi=180, bbox_inches='tight')
    plt.show()
    print(domain_path)
else:
    print('OOD evaluation was disabled in the configuration cell.')


## 10. Option-order robustness

Answer labels move when options are permuted, so label equality cannot measure consistency. We map predicted labels back to choice text and join each perturbation to its parent example. We report perturbed accuracy, the change from parent accuracy, selected-choice consistency, and choice-flip rate.


In [ ]:
from evaluation.robustness import evaluate_robustness

if RUN_ROBUSTNESS:
    robustness_rows = []
    for model_name in MODELS:
        robustness = evaluate_robustness(
            raw_records[(model_name, 'test')],
            raw_records[(model_name, 'shuffled')],
        )
        robustness_rows.append({'model': model_name, **robustness})
        path = METRICS_DIR / f'{model_name}_robustness.json'
        path.write_text(json.dumps(robustness, indent=2) + '\n', encoding='utf-8')

    robustness_table = pd.DataFrame(robustness_rows).set_index('model')
    robustness_table.to_csv(TABLE_DIR / 'option_order_robustness.csv')
    display(robustness_table.style.format({
        column: '{:.4f}'
        for column in robustness_table.columns
        if column != 'paired_records'
    }))

    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6))
    robustness_specs = [
        ('perturbed_accuracy', 'Perturbed accuracy'),
        ('choice_text_consistency', 'Choice-text consistency'),
        ('choice_flip_rate', 'Choice-flip rate'),
    ]
    for ax, (column, title) in zip(axes, robustness_specs):
        values = [robustness_table.loc[name, column] for name in MODELS]
        bars = ax.bar(
            [DISPLAY_NAMES[name] for name in MODELS],
            values,
            color=[COLORS[name] for name in MODELS],
        )
        ax.set_title(title)
        ax.set_ylim(0, max(1.0, max(values) * 1.15))
        ax.grid(axis='y', alpha=0.22)
        ax.bar_label(bars, labels=[f'{value:.3f}' for value in values], padding=3)
    fig.suptitle('Sensitivity to option order')
    fig.tight_layout()
    robustness_path = FIGURE_DIR / 'option_order_robustness.png'
    fig.savefig(robustness_path, dpi=180, bbox_inches='tight')
    plt.show()
    print(robustness_path)
else:
    print('Option-order robustness was disabled in the configuration cell.')


## 11. Inspect one shared test example

Aggregate metrics reveal population behavior, while a shared example verifies the complete inference path. We display the same question for all three models, including the selected answer, verbal confidence, raw option probability, calibrated confidence, abstention decision, parser validity, and original response text.

Change `SAMPLE_INDEX` to inspect another shared record. Sorting by stable ID makes the selection reproducible.


In [ ]:
from IPython.display import HTML, Markdown, display
import html

SAMPLE_INDEX = 0
test_by_model = {
    model_name: {str(record['id']): record for record in raw_records[(model_name, 'test')]}
    for model_name in MODELS
}
calibrated_by_model = {
    model_name: {
        str(record['id']): record
        for record in calibrated_records[(model_name, 'test')]
    }
    for model_name in MODELS
}
shared_ids = sorted(set.intersection(*(set(records) for records in test_by_model.values())))
if not shared_ids:
    raise RuntimeError('No shared test IDs are available.')
sample_id = shared_ids[SAMPLE_INDEX % len(shared_ids)]
reference = test_by_model['base'][sample_id]

choice_lines = '\n'.join(
    f'{chr(ord("A") + index)}. {choice}'
    for index, choice in enumerate(reference['choices'])
)
display(Markdown(
    f'### Shared example `{sample_id}`\n\n'
    f'**Question:** {reference["question"]}\n\n'
    f'```text\n{choice_lines}\n```\n'
    f'**Gold answer:** `{reference["answer_label"]}`'
))

sample_rows = []
for model_name in MODELS:
    raw = test_by_model[model_name][sample_id]
    calibrated = calibrated_by_model[model_name][sample_id]
    prediction = raw.get('prediction') or {}
    calibrated_prediction = calibrated.get('prediction') or {}
    answer = prediction.get('answer')
    sample_rows.append({
        'model': DISPLAY_NAMES[model_name],
        'answer': answer,
        'correct': answer == raw.get('answer_label'),
        'verbal_confidence': prediction.get('confidence'),
        'raw_answer_probability': raw.get('answer_probability'),
        'calibrated_confidence': calibrated.get('answer_probability'),
        'abstain': calibrated_prediction.get('abstain'),
        'schema_valid': prediction.get('schema_valid'),
        'parse_method': prediction.get('parse_method'),
    })

sample_table = pd.DataFrame(sample_rows).set_index('model')
display(sample_table.style.format({
    'verbal_confidence': '{:.4f}',
    'raw_answer_probability': '{:.4f}',
    'calibrated_confidence': '{:.4f}',
}))


In [ ]:
labels = list(calibrated_by_model['base'][sample_id]['option_probabilities'])
fig, axes = plt.subplots(1, len(MODELS), figsize=(15, 4.8), sharey=True)
gold = reference['answer_label']

for ax, model_name in zip(axes, MODELS):
    record = calibrated_by_model[model_name][sample_id]
    probabilities = record['option_probabilities']
    predicted = (record.get('prediction') or {}).get('answer')
    values = [probabilities.get(label, 0.0) for label in labels]
    colors = []
    for label in labels:
        if label == predicted == gold:
            colors.append('#2A9D8F')
        elif label == predicted:
            colors.append('#D95F5F')
        elif label == gold:
            colors.append('#E9C46A')
        else:
            colors.append('#A7ADB4')
    bars = ax.bar(labels, values, color=colors)
    ax.set_title(f'{DISPLAY_NAMES[model_name]}\npredicted={predicted}, gold={gold}')
    ax.set_xlabel('Option label')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.22)
    ax.bar_label(bars, labels=[f'{value:.2f}' for value in values], padding=3)
axes[0].set_ylabel('Calibrated option probability')
fig.suptitle('Probability distribution for one shared example')
fig.tight_layout()
sample_figure_path = FIGURE_DIR / f'sample_{sample_id}.png'
fig.savefig(sample_figure_path, dpi=180, bbox_inches='tight')
plt.show()

for model_name in MODELS:
    prediction = test_by_model[model_name][sample_id].get('prediction') or {}
    raw_text = html.escape(str(prediction.get('raw_text', '')))
    display(HTML(
        f'<h4>{html.escape(DISPLAY_NAMES[model_name])} raw response</h4>'
        f'<pre style="white-space:pre-wrap">{raw_text}</pre>'
    ))


## 12. Produce the report manifest and archive

The manifest records checkpoint identities, dataset scope, calibration objects, summary tables, and artifact locations. The archive provides one transportable copy of the complete report directory. Raw predictions remain the source of truth for recomputing every metric.

A final report should use `RUN_MODE = 'full'`. We verify the full run by checking that all audit rows pass, every model has the same in-domain IDs, the robustness join includes every evaluated parent, and the manifest points to the intended remote checkpoints.


In [ ]:
import shutil
from datetime import datetime, timezone

report_manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_mode': RUN_MODE,
    'dataset_repo': DATASET_REPO,
    'base_model': BASE_MODEL,
    'renderer': RENDERER,
    'evaluation_limit': EVALUATION_LIMIT,
    'run_ood': RUN_OOD,
    'run_robustness': RUN_ROBUSTNESS,
    'target_coverage': TARGET_COVERAGE,
    'models': MODELS,
    'calibrations': calibrations,
    'in_domain_summary': summary_table.reset_index().to_dict(orient='records'),
    'verification': verification_table.to_dict(orient='records'),
    'audit': audit_table.to_dict(orient='records'),
}
manifest_path = REPORT_ROOT / 'report_manifest.json'
manifest_path.write_text(json.dumps(report_manifest, indent=2) + '\n', encoding='utf-8')

archive_base = REPORT_ROOT.parent / f'{REPORT_ROOT.name}_{RUN_MODE}_bundle'
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', REPORT_ROOT))

artifact_rows = []
for path in sorted(REPORT_ROOT.rglob('*')):
    if path.is_file():
        artifact_rows.append({
            'artifact': str(path.relative_to(REPORT_ROOT)),
            'size_kb': path.stat().st_size / 1024,
        })
display(pd.DataFrame(artifact_rows).style.format({'size_kb': '{:.1f}'}))
print(f'Manifest: {manifest_path}')
print(f'Archive:  {archive_path}')


## Reading the final result

We first compare accuracy and format validity to determine whether training improved task performance and contract adherence. We then compare calibrated ECE, multiclass NLL, and Brier score to assess probability quality. We use AURC and selective accuracy to determine whether confidence ranks errors effectively. We inspect OOD deltas and option-order consistency to detect brittle gains. Finally, we inspect shared examples to connect aggregate changes to concrete response behavior.

We treat a trained model as an improvement when gains are supported across several independent views. A higher accuracy accompanied by lower ECE, lower NLL, lower AURC, stable OOD behavior, and stronger option-order consistency provides substantially stronger evidence than accuracy alone.
